In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

DATA_PATH = "your_dataset.csv"

ID_COL = "SampleID"
TIME_COL = "Time_s"
TIME_H_COL = "Time_h"
PHASE_COL = "Phase"
LABEL_COL = "Treatment"

SENSORS = [
    "TGS2600",
    "TGS2602",
    "TGS822",
    "MQ3",
    "MQ135",
    "MQ138",
    "MiCS_NO2",
    "MiCS_NH3",
    "MiCS_CO"
]

DATA_URL = "https://drive.google.com/file/d/1MH9Qu8hO1uS3eGI8jUFTTQynMYU0NAI9/view?usp=drive_link"

df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)

df = df.dropna(
    subset=SENSORS + [
        ID_COL,
        TIME_H_COL,
        PHASE_COL,
        LABEL_COL
    ]
).copy()

for col in SENSORS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.dropna(
    subset=SENSORS
).copy()

df = df.sort_values(
    [ID_COL, TIME_H_COL, PHASE_COL, TIME_COL]
).reset_index(drop=True)

print("\nTreatment distribution:")
print(df[LABEL_COL].value_counts())

print("\nPhase distribution:")
print(df[PHASE_COL].value_counts())


df["infected"] = np.nan

df.loc[
    df[LABEL_COL] == "Control",
    "infected"
] = 0

df.loc[
    df[LABEL_COL].isin(
        ["Low", "Medium", "High"]
    ),
    "infected"
] = 1


df_model = df[
    df[LABEL_COL].isin(
        ["Control", "Low", "Medium", "High"]
    )
].copy()

df_model["infected"] = (
    df_model["infected"]
    .astype(int)
)


print("\nBinary class distribution:")
print(
    df_model["infected"].value_counts()
)


GROUP_COLUMNS = [
    ID_COL,
    TIME_H_COL,
    PHASE_COL
]


feature_frames = []

for sensor in SENSORS:

    temp = (
        df_model
        .groupby(GROUP_COLUMNS)[sensor]
        .agg(
            [
                "mean",
                "std",
                "min",
                "max"
            ]
        )
        .reset_index()
    )

    temp = temp.rename(
        columns={
            "mean": f"{sensor}_mean",
            "std": f"{sensor}_std",
            "min": f"{sensor}_min",
            "max": f"{sensor}_max"
        }
    )

    feature_frames.append(temp)


features = feature_frames[0]

for temp in feature_frames[1:]:

    features = features.merge(
        temp,
        on=GROUP_COLUMNS,
        how="inner"
    )


metadata = (
    df_model[
        GROUP_COLUMNS +
        [LABEL_COL, "infected"]
    ]
    .drop_duplicates(
        subset=GROUP_COLUMNS
    )
)


features = features.merge(
    metadata,
    on=GROUP_COLUMNS,
    how="inner"
)


features = features.fillna(0)


FEATURE_COLUMNS = [
    col
    for col in features.columns
    if col not in [
        ID_COL,
        TIME_H_COL,
        PHASE_COL,
        LABEL_COL,
        "infected"
    ]
]


X = features[FEATURE_COLUMNS]

y = features["infected"]

groups = features[ID_COL]


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)


train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)


X_train = X.iloc[train_idx]

X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]

y_test = y.iloc[test_idx]


model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


print("\nTraining model...")

model.fit(
    X_train,
    y_train
)

print("Training complete.")


y_pred = model.predict(
    X_test
)

y_prob = model.predict_proba(
    X_test
)[:, 1]


print("\n==============================")
print("MODEL RESULTS")
print("==============================")

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

try:

    print(
        "\nROC-AUC:",
        roc_auc_score(
            y_test,
            y_prob
        )
    )

except Exception:
    pass


feature_importance = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLUMNS
).sort_values(
    ascending=False
)


print(
    "\nTop features:"
)

print(
    feature_importance.head(20)
)


sensor_importance = {}

for sensor in SENSORS:

    total = 0

    for stat in [
        "mean",
        "std",
        "min",
        "max"
    ]:

        feature_name = (
            f"{sensor}_{stat}"
        )

        if feature_name in feature_importance.index:

            total += feature_importance[
                feature_name
            ]

    sensor_importance[
        sensor
    ] = total


sensor_importance = pd.Series(
    sensor_importance
).sort_values(
    ascending=False
)


print(
    "\nSensor importance:"
)

print(
    sensor_importance
)


joblib.dump(
    model,
    "olfact_model.pkl"
)

joblib.dump(
    FEATURE_COLUMNS,
    "feature_columns.pkl"
)

joblib.dump(
    SENSORS,
    "sensors.pkl"
)

joblib.dump(
    {
        "id_col": ID_COL,
        "time_col": TIME_COL,
        "time_h_col": TIME_H_COL,
        "phase_col": PHASE_COL,
        "label_col": LABEL_COL,
        "group_columns": GROUP_COLUMNS,
        "probability_threshold": 0.70
    },
    "model_config.pkl"
)

feature_importance.to_csv(
    "feature_importance.csv"
)

sensor_importance.to_csv(
    "sensor_importance.csv"
)


print("\n==============================")
print("FILES CREATED")
print("==============================")

print("olfact_model.pkl")
print("feature_columns.pkl")
print("sensors.pkl")
print("model_config.pkl")
print("feature_importance.csv")
print("sensor_importance.csv")


Dataset shape: (4410000, 15)

Treatment distribution:
Treatment
Control       882000
Low           882000
Medium        882000
High          882000
Mechanical    882000
Name: count, dtype: int64

Phase distribution:
Phase
acquisition    2520000
recovery       1260000
baseline        630000
Name: count, dtype: int64

Binary class distribution:
infected
1    2646000
0     882000
Name: count, dtype: int64

Training model...
Training complete.

MODEL RESULTS
Accuracy: 0.7797619047619048

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.35      0.43       240
           1       0.82      0.92      0.86       768

    accuracy                           0.78      1008
   macro avg       0.69      0.63      0.65      1008
weighted avg       0.76      0.78      0.76      1008


Confusion Matrix:
[[ 83 157]
 [ 65 703]]

ROC-AUC: 0.7533528645833333

Top features:
TGS822_std      0.054521
MQ135_std       0.053988
MQ3_std         0.051829
